# 00 · Access the Model

This is the warm-up to confirm your environment can reach a Granite model. It introduces `get_llm()`, the helper every notebook in this workshop uses to connect to Granite: a local Ollama server first, Replicate as the hosted fallback.

Every notebook is self-contained, so each one defines its own copy of `get_llm()` near the top rather than importing it from a shared package. If a model connection ever misbehaves, that cell is the place to look.

# Steps

## Step 1. Set up your environment


You can run this notebook in [Colab](https://colab.research.google.com/), or download it to your system and [run the notebook locally](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started_with_Jupyter_Locally/Getting_Started_with_Jupyter_Locally.md).

## Step 2. Set up a Granite AI model instance

This notebook requires IBM Granite models to be served by an AI model runtime so that the models can be invoked or called. This notebook can use a locally accessible [Ollama](https://ollama.com) server to serve the models, or the [Replicate](https://replicate.com) cloud service.

* See [Getting Started with Replicate](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_Replicate.ipynb) for information on getting ready to use Replicate. 
* See [Getting Started with Ollama](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_Ollama.ipynb) for information on getting ready to use Ollama.

Prior to running this notebook, you should have either started a local Ollama server on your computer, or setup Replicate access and obtained an [API token](https://replicate.com/account/api-tokens).

## Step 3. Install relevant libraries and connect to Granite

We only need a few libraries for this warm-up: the Granite community helpers, LangChain, and the two model backends (Ollama, Replicate).

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    langchain \
    langchain_ollama \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git"
! echo "::endgroup::"

Now we will define `get_llm()` and use it to create the LangChain object for the Granite model. It tries a locally accessible Ollama server first, and falls back to Replicate otherwise.

In [ ]:
import requests
from ibm_granite_community.notebook_utils import get_env_var


def ollama_has_model(host: str, model: str) -> bool:
    """True only if `host` answers AND `model` is already pulled there.

    Checking reachability alone is not enough: a running Ollama without the
    requested tag would otherwise trigger an on-demand pull, which needs
    network access we don't have on the day.
    """
    try:
        tags = requests.get(f"{host}/api/tags", timeout=2).json()
        return model in {m["name"] for m in tags.get("models", [])}
    except requests.RequestException:
        return False


def get_llm(
    model: str | None = None,
    *,
    backend: str | None = None,
    stop: list[str] | None = None,
    repetition_penalty: float | None = None,
    **kwargs,
):
    """Return a LangChain chat model for Granite.

    Args:
        model: An Ollama tag (e.g. "granite4.2:3b") or a Replicate model path
            (e.g. "ibm-granite/granite-4.2-8b"). Defaults to the `GRANITE_MODEL`
            env var (or "granite4.2:3b") for Ollama, and to `REPLICATE_MODEL`
            (or "ibm-granite/granite-4.2-8b") for Replicate.
        backend: Force "ollama" or "replicate"; omit to auto-resolve
            (Ollama if the tag is pulled locally, Replicate otherwise).
        stop: Stop sequences, passed the way each backend expects them.
        repetition_penalty: Penalty for repeated tokens, passed the way each
            backend expects it.
        **kwargs: Extra keyword arguments passed through to the chat model.
    """
    host = get_env_var("OLLAMA_HOST", "http://127.0.0.1:11434")
    ollama_model = model or get_env_var("GRANITE_MODEL", "granite4.2:3b") or "granite4.2:3b"

    if backend != "replicate" and (backend == "ollama" or ollama_has_model(host, ollama_model)):
        from langchain_ollama import ChatOllama

        return ChatOllama(
            model=ollama_model,
            base_url=host,
            num_predict=8192,  # Set the maximum number of tokens to generate as output.
            stop=stop,
            repeat_penalty=repetition_penalty,
            **kwargs,
        )

    from langchain_replicate import ChatReplicate

    # Only a Replicate model path ("owner/name") can be sent to Replicate; an Ollama tag cannot.
    if model and "/" in model:
        replicate_model = model
    else:
        replicate_model = get_env_var("REPLICATE_MODEL", "ibm-granite/granite-4.2-8b") or "ibm-granite/granite-4.2-8b"

    # ChatReplicate silently ignores unknown constructor arguments, so generation
    # settings must go in `model_kwargs`, which is sent as the prediction input.
    model_kwargs = {
        "max_completion_tokens": 8192,  # Set the maximum number of tokens to generate as output.
        "chat_template_kwargs": {"enable_thinking": False},
    }
    if stop:
        model_kwargs["stop"] = stop
    if repetition_penalty is not None:
        model_kwargs["repetition_penalty"] = repetition_penalty

    return ChatReplicate(
        model=replicate_model,
        replicate_api_token=get_env_var("REPLICATE_API_TOKEN"),
        model_kwargs=model_kwargs,
        **kwargs,
    )

In [ ]:
llm = get_llm()
print(f"Connected via {type(llm).__name__}, model={getattr(llm, 'model', None)!r}")

If that printed `ChatOllama`, you're using your local Ollama server; if it printed `ChatReplicate`, you're using the Replicate cloud service instead. Either is fine -- the rest of this workshop works the same way regardless of which one you're on.

## Step 4. Run a simple prompt

Let's confirm the model actually answers, with nothing else in the way -- no tools, no agent loop, just a single prompt and a response.

In [ ]:
response = llm.invoke("In one sentence, what is IBM Granite?")
print(response.content)

If you got a one-sentence answer above, your environment is ready. Head to [`01_Function_Calling_Agent.ipynb`](01_Function_Calling_Agent.ipynb) next, where `llm` starts being asked to call tools.